# 01 EDA and leakage review (cities A and B)
Owner: Viktoriia Markus

Region **Australia** (AUD). **A = Sydney, B = Melbourne**, snapshots of 2026-09-16. City C is not loaded here until week 1 is finished.

Reads the pipeline's tables (`bronze.listings_raw`, `silver.listings`), so run `00_run_pipeline` for both cities first. Nothing here writes to a table.

## Summary
*Numbers in [brackets] come from the last cell, `Facts for the summary`, after a full run.*

**What we now know**
- **Grain:** one row = one listing as it was on the snapshot date; `id` is unique in silver (duplicates removed: [n]).
- **Size:** Sydney [n] listings, Melbourne [n].
- **No price for a large share:** about **15% in Sydney and 26% in Melbourne** have no valid price (`is_valid_price = false`). Section 2.1 shows these are mostly inactive listings (blocked calendar, no recent reviews), so the price is *not applicable*, not *unknown*. They are **excluded** from every price analysis and from the model; they stay in silver for listing counts.
- **Other gaps stay NULL** and are structural, not random: review scores are NULL exactly when there are no reviews; bedrooms/beds are NULL mostly for rooms and studios; host response fields are NULL together (host never answered requests).
- **Price** is strongly right-skewed (median [x] AUD in Sydney, [x] in Melbourne); on a log scale it is roughly normal within a room type, so the model should predict `log(price)`.
- **Outliers:** [n] typed-in placeholder prices (9999, 99999...) to drop from training; [n] high/low outliers that look real (large homes); [n] long-stay listings (`minimum_nights > 30`) whose nightly price means something else.
- **Leakage:** `estimated_revenue_l365d` = price × estimated occupancy (holds for [x]% of rows) and is NULL whenever price is NULL. It must never be a feature.

**What we still don't know**
- Whether the advertised snapshot price is what guests pay (no seasonality, discounts or cleaning fee).
- Whether long-stay listings quote a real nightly rate or a monthly rate split by nights.
- How well neighbourhood-based features transfer to a city whose neighbourhoods the model has never seen (city C).

In [ ]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from airbnb import config, eda
importlib.reload(eda)

pd.set_option("display.max_columns", 60)
dbutils.widgets.text("cities", ",".join(config.TRAIN_CITIES))
CITIES = [c.strip() for c in dbutils.widgets.get("cities").split(",") if c.strip()]

# City C stays unseen until the week-1 pipeline and model are tagged `week1-final`.
unseen = set(CITIES) - set(config.TRAIN_CITIES)
assert not unseen, f"{unseen} is not a training city. Compare it in 03_city_c, in week 2."

df_all = eda.load_silver(spark, CITIES)                                   # every listing
df = df_all[df_all["is_valid_price"].fillna(False).astype(bool)].copy()   # price analysis: valid prices only
print(f"silver rows: {len(df_all)}, with a valid price: {len(df)}")

## 1. The dataset
**Grain: one row = one listing** as it looked on the snapshot date. Source: Inside Airbnb `listings.csv.gz` (detailed listings), latest snapshot per city.
Bronze is the file as it came (every column as text); silver is one typed row per `id`.

In [ ]:
bo = eda.bronze_overview(spark, CITIES).set_index("city")
so = eda.silver_overview(df_all)
overview = bo.join(so)
overview["bronze_minus_silver"] = overview["bronze_rows"] - overview["silver_rows"]
display(overview.reset_index())

In [ ]:
raw_cols = eda.bronze_columns(spark)
print(f"Raw file: {len(raw_cols)} columns. Silver keeps {len(df_all.columns)} (typed + derived).")
print("Raw columns not carried to silver:", sorted(set(raw_cols) - set(df_all.columns)))

## 2. Columns we use
Types come from silver. Identifiers, URLs, photos and scrape metadata are not profiled (they are not features, see section 4).

In [ ]:
PROFILE_COLS = [
    "price", "room_type", "property_type", "accommodates", "bedrooms", "beds", "bathrooms",
    "bathrooms_text", "is_shared_bath", "amenities_count", "neighbourhood_cleansed", "latitude", "longitude",
    "host_tenure_days", "host_is_superhost", "host_response_time", "host_response_rate", "host_acceptance_rate",
    "host_listings_count", "host_identity_verified", "calculated_host_listings_count",
    "minimum_nights", "maximum_nights", "instant_bookable", "has_availability",
    "availability_30", "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "first_review", "last_review", "reviews_per_month",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_location", "review_scores_value",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
]
PROFILE_COLS = [c for c in PROFILE_COLS if c in df_all.columns]
spark_types = dict(spark.table(config.LISTINGS_TABLE).dtypes)
types = pd.DataFrame({
    "silver_type": [spark_types.get(c) for c in PROFILE_COLS],
    "distinct": df_all[PROFILE_COLS].nunique().values,
    "null_%_all_rows": (df_all[PROFILE_COLS].isna().mean() * 100).round(1).values,
    "example": [df_all[c].dropna().iloc[0] if df_all[c].notna().any() else None for c in PROFILE_COLS],
}, index=PROFILE_COLS)
display(types.reset_index(names="column"))

### 2.1 Listings without a price (`is_valid_price = false`)
About 15% in Sydney and 26% in Melbourne. Before dropping them, check **why** they have no price: compare them with priced listings.

In [ ]:
display(eda.why_no_price(df_all).reset_index())

In [ ]:
fig, axes = plt.subplots(1, len(CITIES), figsize=(7 * len(CITIES), 4), squeeze=False, sharey=True)
for ax, city in zip(axes[0], CITIES):
    g = df_all[df_all.city == city]
    valid = g["is_valid_price"].fillna(False).astype(bool)
    ax.hist([g.loc[valid, "availability_365"], g.loc[~valid, "availability_365"]], bins=37,
            label=["valid price", "no price"], density=True)
    ax.set_title(f"{city}: listings without a price are mostly blocked (availability 0)")
    ax.set_xlabel("availability_365"); ax.legend()
plt.tight_layout(); plt.show()

**Reading (check against the tables above):** listings without a price have far more `availability_365 = 0`, `has_availability` false and no reviews in the last year. They are inactive or delisted, so Inside Airbnb has no advertised price for them.
**Decision:** keep them in silver (they still count as listings for the dashboard), exclude them from price analysis, the baseline and the model (`is_valid_price = true` only). Melbourne's higher share means more inactive listings, not worse data.

### 2.2 Other missing values (listings with a valid price)
Missing values stay **NULL** in silver. Filling them is a modelling choice, learned on the training split only.

In [ ]:
miss = eda.missing_table(df, PROFILE_COLS)
display(miss[miss.max(axis=1) > 0].rename(columns=lambda c: f"{c} % NULL").reset_index(names="column"))

In [ ]:
# Where the gaps are: rows (sorted by room type, then number of reviews) x columns. Black = NULL.
cols = miss.index[miss.max(axis=1) > 0].tolist()
fig, axes = plt.subplots(1, len(CITIES), figsize=(7 * len(CITIES), 6), squeeze=False)
for ax, city in zip(axes[0], CITIES):
    g = df[df.city == city].sort_values(["room_type", "number_of_reviews"])
    ax.imshow(g[cols].isna().to_numpy(), aspect="auto", cmap="Greys", interpolation="nearest")
    ax.set_xticks(range(len(cols)), cols, rotation=90, fontsize=7)
    ax.set_yticks([]); ax.set_title(f"{city}: NULLs (rows sorted by room type, reviews)")
plt.tight_layout(); plt.show()

In [ ]:
# Missing for a reason? Missing rate when the condition is true vs false. A big gap = a pattern.
display(eda.missing_patterns(df))

**Reading (check against the output):**
- **Review scores, `reviews_per_month`, `first_review`** are NULL when `number_of_reviews = 0`: *not applicable*. The model gets a `has_reviews` flag instead of a mean fill.
- **`bedrooms`, `beds`** are NULL mostly for private/shared rooms and small listings (studios): the host left it empty. Fill on train only, e.g. median by `room_type` × `accommodates`.
- **`bathrooms`** is filled from `bathrooms_text` in silver; what is left NULL has no text either.
- **Host response fields** are NULL together: the host has had no requests to answer. Treat as their own category.

### 2.3 Price distribution (valid prices)

In [ ]:
display(eda.price_summary(df).reset_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for city in CITIES:
    p = df.loc[df.city == city, "price"]
    axes[0].hist(p.clip(upper=p.quantile(.99)), bins=80, alpha=.5, label=city)
    axes[1].hist(np.log10(p), bins=80, alpha=.5, label=city)
axes[0].set_title("Price, AUD per night (clipped at the 99th percentile): long right tail"); axes[0].set_xlabel("AUD")
axes[1].set_title("log10(price): close to normal, so the model predicts log price"); axes[1].set_xlabel("log10 AUD")
for a in axes: a.legend()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(CITIES), figsize=(7 * len(CITIES), 4), squeeze=False, sharey=True)
for ax, city in zip(axes[0], CITIES):
    g = df[df.city == city]
    groups = sorted(g.room_type.dropna().unique())
    ax.boxplot([g.loc[g.room_type == r, "price"] for r in groups], whis=3)
    ax.set_xticks(range(1, len(groups) + 1), groups)
    ax.set_yscale("log"); ax.set_title(f"{city}: price by room type (log scale)"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
# Median price by room type x accommodates: this grid is the baseline the model must beat
grid = df.groupby(["city", "room_type", "accommodates"])["price"].median().unstack(["city", "room_type"])
display(grid.round(0).reset_index())

### 2.4 Availability distribution (valid prices)

In [ ]:
display(eda.availability_summary(df).reset_index())
fig, ax = plt.subplots(figsize=(10, 4))
for city in CITIES:
    ax.hist(df.loc[df.city == city, "availability_365"], bins=73, alpha=.5, label=city)
ax.set_title("Days available in the next 365 (priced listings)"); ax.set_xlabel("availability_365"); ax.legend()
plt.show()

### 2.5 Outliers and how we classify them
Fences on **log price**, per city × room type, at Q1 − 3·IQR and Q3 + 3·IQR (groups under 30 listings use the city's fences). Wide on purpose: separate *impossible* from *rare but real*.

| Class | Meaning | Pipeline / model |
|---|---|---|
| `placeholder` | above the high fence and a typed-looking number (9999, 10000, 99999) | flag, drop from train and test |
| `high_outlier` / `low_outlier` | outside the fences, plausible after inspection | keep, flag `is_price_outlier`; log price limits their pull |
| `long_stay` | `minimum_nights > 30`: a monthly rental | keep, flag, exclude from the model |
| `normal` | everything else | keep |

In [ ]:
df = eda.classify_price(df)
display(pd.crosstab(df["city"], df["price_class"], margins=True).reset_index())

In [ ]:
# Inspect by hand: luxury homes or errors?
show = ["city", "id", "price", "price_class", "price_high_fence", "room_type", "property_type",
        "accommodates", "bedrooms", "minimum_nights", "availability_365", "number_of_reviews", "name"]
show = [c for c in show if c in df]
display(df[df.price_class.isin(["placeholder", "high_outlier"])].sort_values("price", ascending=False)[show].head(30))
display(df[df.price_class == "low_outlier"].sort_values("price")[show].head(15))

In [ ]:
# Other values outside what is physically plausible (counts per city)
display(eda.other_outliers(df).reset_index())

## 3. Cleaning the pipeline must do
Each rule, its reason, and whether `silver.py` already does it.

| # | Rule | Reason (section) | In silver.py |
|---|---|---|---|
| 1 | Read every column as text in bronze | the file mixes `"$1,234.00"`, `"95%"`, `"t"/"f"`; guessing types loses values | ✅ bronze |
| 2 | One row per `id` (keep latest `last_scraped`) | grain is one listing (1) | ✅ |
| 3 | `price`: strip `$`, `,` → double | target arrives as text (2.3) | ✅ |
| 4 | `is_valid_price = price IS NOT NULL AND price > 0`; keep the rows, exclude from price work | missing price = inactive listing (2.1) | ✅ flag; filter in features/model |
| 5 | Empty string and `N/A` → NULL; leave NULLs as NULL | filling is a model step, learned on train only (2.2) | ✅ |
| 6 | Percentages → 0..1, `t`/`f` → boolean, dates → date | text types (2) | ✅ |
| 7 | `bathrooms` from `bathrooms_text` when empty; `is_shared_bath` | `bathrooms` often empty (2.2) | ✅ |
| 8 | `host_tenure_days` against `snapshot_date`, never `current_date()` | reproducible on any day | ✅ |
| 9 | Flag `is_long_stay = minimum_nights > 30` | nightly price means something else (2.5) | ➕ to add |
| 10 | Flag placeholder prices (above fence and 9999/10000/99999 pattern) | typed-in values (2.5) | ➕ to add |
| 11 | `has_reviews = number_of_reviews > 0` | review NULLs are structural (2.2) | ➕ to add (or in features) |
| 12 | Check `room_type` against the 4 known values, log unseen ones | a new city can bring new categories (week 2) | ➕ to add to validation |
| 13 | Drop leakage, id and metadata columns from the **feature** table, not from silver | section 4 | ➕ `features.py` |

## 4. Leakage review
The target is the advertised nightly `price` on the snapshot date. A column leaks if it is **computed from the price** or **states it**.
Four tests: names, correlations, an exact check of the formula behind the suspicious column, and the free text.

In [ ]:
print("Raw columns whose name points to money:   ", eda.leakage_name_scan(raw_cols))
print("Silver columns whose name points to money:", eda.leakage_name_scan(df_all.columns))

In [ ]:
# Spearman correlation with price (valid prices). High alone is NOT leakage: accommodates is a real driver.
# It only says which columns to examine.
display(eda.leakage_correlations(df).head(20).reset_index(names="column"))

In [ ]:
# The decisive check: is estimated_revenue_l365d exactly price x estimated_occupancy_l365d?  (all silver rows)
display(eda.revenue_identity(df_all))

In [ ]:
# Does the free text state the price? % of listings with "$<digits>" in name / description (bronze)
display(eda.price_in_text(spark, CITIES))

### Conclusion
**Leakage: never a feature** (`features.LEAKAGE_COLS`)

| Column | Why |
|---|---|
| `estimated_revenue_l365d` | Inside Airbnb computes it as price × estimated booked nights: the identity check holds for almost every row, and it is NULL exactly when price is NULL. Using it is using the target. |
| `price`, `is_valid_price`, anything derived from price (price per guest, `price_class`) | the target itself or a function of it |

**Not leakage, but never features** (`features.NON_FEATURE_COLS`): `id`, `host_id`, `last_scraped`, `snapshot_date`, `_ingested_at`, `license`, `name` and the raw URLs / scrape metadata (not in silver). They describe the scrape, not the listing, or let a model memorise rows.

**Free text** (`name`, `description`): a small share writes the price in the text (table above). Raw text is not used; if it ever is, strip `$<number>` first.

**Allowed, with care** (`features.CAUTION_COLS`): `estimated_occupancy_l365d`, `availability_*`, `number_of_reviews_ltm`. Not computed from price (occupancy comes from reviews and minimum nights), but they react to it over time (expensive listings book less). Out of the first model; added only if they clearly help on the test set.

**Transfers badly to city C:** `neighbourhood_cleansed`: its values do not exist in another city. Prefer room type, size, amenities, distance to the city centre from latitude/longitude.

## 5. For city C (week 2, with P1)
The same helpers profile a new city once the pipeline has run on it. In `03_city_c`, after the unchanged first run:
```python
cmp = eda.compare_to_training(spark, "brisbane")
for name, table in cmp.items(): print(name); display(table)
```
It lists raw columns empty or new in C, unseen `room_type` / `property_type` / neighbourhoods, the share without a price and why, price and size distributions next to A and B, NULL rates and outlier classes. Each difference becomes a *pipeline fix* or a *model change* in `docs/city_c_report.md`.

### Facts for the summary

In [ ]:
facts = {}
rev = eda.revenue_identity(df_all).set_index("city")
for city in CITIES:
    a, v = df_all[df_all.city == city], df[df.city == city]
    facts[city] = {
        "snapshot": str(a["snapshot_date"].max().date()),
        "silver rows": len(a),
        "bronze - silver (duplicates / no id)": int(overview.loc[city, "bronze_minus_silver"]),
        "no valid price %": round((~a["is_valid_price"].fillna(False).astype(bool)).mean() * 100, 1),
        "valid-price listings": len(v),
        "median price AUD": float(v["price"].median()),
        "placeholder prices": int((v.price_class == "placeholder").sum()),
        "high / low outliers": int(v.price_class.isin(["high_outlier", "low_outlier"]).sum()),
        "long stay": int((v.price_class == "long_stay").sum()),
        "revenue identity holds %": rev.loc[city].get("identity_holds_within_1pct_%"),
    }
display(pd.DataFrame(facts).reset_index(names="fact"))